# Variational linear solve with LCU

All 17 source code stages, the original three-term 8 by 8 matrix, three system and two selector bits, all nine ansatz angles, the conditional cost with 204800 actual draws per query, a cap of 2000 objective queries, and comparison of the final state with the direct solve.

Native COBYLA replaces the unpinned hosted optimization, with seed 1000 for the source initial angles and an independent local sampling stream. Preserve the last executed parameter binding returned by the source wrapper. Statevector readout yields the actual final amplitudes. The final phase is aligned to the source last-component convention. The routine counts postselection events and reports the actual trace; no historical optimizer trajectory, convergence, or speedup claim.

## Use the existing native runtime

In [ ]:
from pathlib import Path

from IPython.display import Image, display


## Load the native LCU and optimizer

In [ ]:
from examples.synthesis.algorithms.quantum_linear_solvers.vqls.vqls_with_lcu.overview.plots import (
    save_comparison,
    save_trace,
)
from examples.synthesis.algorithms.quantum_linear_solvers.vqls.vqls_with_lcu.overview.quantum import (
    MATRIX,
    TARGET,
    build_objective,
)
from examples.synthesis.algorithms.quantum_linear_solvers.vqls.vqls_with_lcu.overview.workflow import (
    BUDGET,
    SHOTS,
    run_workflow,
)

folder = Path("artifacts/development/vqls-lcu")


## Define the conditional objective

In [ ]:
print("Cost = 1 - count(all zero) / count(selector zero)")


## Inspect the original three-term operator

In [ ]:
print("Operator diagonal:", MATRIX.tolist())


## Retain the three-layer ansatz

In [ ]:
print("Nine rotations, with two three-edge CZ layers")


## Check the parameter count

In [ ]:
angles = (0.1,) * 9
assert len(angles) == 9


## Compose the objective program

In [ ]:
objective = build_objective(angles)


## Inspect the checked execution resources

In [ ]:
print(objective.report["resources"])


## Run the original finite-shot optimization

In [ ]:
result = run_workflow()
last = result.queries[-1]
print("Actual objective queries:", len(result.queries))
print("Objective budget and shots per query:", BUDGET, SHOTS)
print("Total actual objective draws:", len(result.queries) * SHOTS)
print("Last auxiliary-zero and all-zero counts:", last.auxiliary_zero, last.all_zero)
print("Last sampled cost:", last.cost)
print("Converged and budget stop:", result.converged, result.budget_stop)
display(Image(filename=str(save_trace(result, folder / "trace.png"))))


## Inspect the selected solution program

In [ ]:
print(result.solution.report["resources"])


## Read the state from the executed circuit

In [ ]:
assert result.amplitudes.shape == (8,)


## Report the source phase convention

In [ ]:
print("Solution amplitudes:", result.amplitudes.real.tolist())


## Read the probabilities

In [ ]:
print("Variational probabilities:", result.amplitudes.abs().square().tolist())


## Inspect the independent linear system

In [ ]:
print("Matrix diagonal and target:", MATRIX.tolist(), TARGET.tolist())


## Read the normalized direct solution

In [ ]:
print("Direct solution amplitudes:", result.reference.real.tolist())


## Compare overlap and residual

In [ ]:
print("Solution fidelity:", result.fidelity)
print("Normalized residual:", result.normalized_residual)


## Plot the two solution distributions

In [ ]:
display(Image(filename=str(save_comparison(result, folder / "solution.png"))))
